<a href="https://colab.research.google.com/github/xKABIRAJ/gz/blob/main/WeatherPred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.metrics import mean_squared_error
from datetime import datetime, timedelta

In [ ]:
API_KEY = '9e31df8c517546edc5306fc58bdcca89'
BASE_URL = 'https://api.openweathermap.org/data/2.5/'


In [ ]:
def get_current_weather(city):
  url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"
  response = requests.get(url)
  data = response.json()
  if response.status_code != 200:
      print(f"Error fetching weather for {city}: {data.get('message', 'Unknown error')}")
      return None
  return{
      'city':data['name'],
      'current_temp' :round(data['main']['temp']),
      'feels_like' : round(data['main']['feels_like']),
      'temp_min': round(data['main']['temp_min']),
      'temp_max' : round(data['main']['temp_max']),
      'humidity' : round(data['main']['humidity']),
      'pressure' : round(data['main']['pressure']),
      'description' : data['weather'][0]['description'],
      'country': data['sys']['country'],
      'wind_speed': round(data['wind']['speed']) if 'wind' in data and 'speed' in data['wind'] else 0,
      'wind_deg': round(data['wind']['deg']) if 'wind' in data and 'deg' in data['wind'] else 0,
  }

In [ ]:
def read_historical_data(filename):
  df = pd.read_csv(filename)
  df = df.dropna()
  df = df.drop_duplicates()
  return df

In [ ]:
def prepare_data(data):
  data_copy = data.copy()
  le_wind_dir = LabelEncoder()
  data_copy['WindGustDir'] = le_wind_dir.fit_transform(data_copy['WindGustDir'])
  le_rain_tomorrow = LabelEncoder()
  data_copy['RainTomorrow'] = le_rain_tomorrow.fit_transform(data_copy['RainTomorrow'])

  x = data_copy[['MinTemp','MaxTemp','WindGustDir','WindGustSpeed','Humidity','Pressure','Temp']]
  y = data_copy['RainTomorrow'] #Target variable

  return x, y, le_wind_dir, le_rain_tomorrow

In [ ]:
def train_rain_model(x,y):
  x_train,x_test,y_train,y_test = train_test_split(x,y, test_size = 0.2, random_state=42 )
  model = RandomForestClassifier(n_estimators=100, random_state =42)
  model.fit(x_train,y_train)

  y_pred = model.predict(x_test)

  from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

  print("\n--- Rain Prediction Model Evaluation ---")
  print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
  print(f"Precision: {precision_score(y_test, y_pred):.4f}")
  print(f"Recall: {recall_score(y_test, y_pred):.4f}")
  print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
  print("--------------------------------------")

  return model

In [ ]:
def prepare_regression_data(data, feature):
  x,y = [],[]

  for i in range(len(data)-1):
    x.append(data[feature].iloc[i])

    y.append(data[feature].iloc[i+1])

  x = np.array(x).reshape(-1,1)
  y = np.array(y)

  return x,y

In [ ]:
def train_regession_model(x,y):
  model = RandomForestRegressor (n_estimators = 100, random_state = 42)
  model.fit(x,y)

  return model

In [ ]:
def predict_future(model, current_value, num_steps=5):
  predictions = [current_value]

  for i in range(num_steps):
    next_value = model.predict(np.array([predictions[-1]]).reshape(1, -1))

    predictions.append(next_value[0])

  return predictions[1:]

In [ ]:
def weather_view():
  city = input("Enter any city name: ")
  current_weather = get_current_weather(city)
  if current_weather is None:
      print(f"Could not retrieve weather for {city}. Exiting.")
      return

  historical_data = read_historical_data('/content/weather.csv')

  # Get features, target, and the LabelEncoder for WindGustDir and RainTomorrow
  X_historical, y_historical, le_wind_dir, le_rain_tomorrow = prepare_data(historical_data)

  rain_model = train_rain_model(X_historical, y_historical)

  wind_deg_val = current_weather['wind_deg']
  compass_points = [
      ("N", 0, 11.25), ("NNE", 11.25, 33.75), ("NE", 33.75, 56.25),
      ("ENE", 56.25, 78.75), ("E", 78.75, 101.25), ("ESE", 101.25, 123.75),
      ("SE", 123.75, 146.25), ("SSE", 146.25, 168.75), ("S", 168.75, 191.25),
      ("SSW", 191.25, 213.75), ("SW", 213.75, 236.25), ("WSW", 236.25, 258.75),
      ("W", 258.75, 281.25), ("WNW", 281.25, 303.75), ("NW", 303.75, 326.25),
      ("NNW", 326.25, 348.75), ("N", 348.75, 360)
  ]

  wind_gust_dir_str = "N" # Default
  for direction, start, end in compass_points:
      if start <= wind_deg_val < end:
          wind_gust_dir_str = direction
          break
  # Handle the wrap-around for North (0-11.25 and 348.75-360)
  if wind_deg_val >= 348.75 or wind_deg_val < 11.25:
      wind_gust_dir_str = "N"

  # Encode the current wind direction using the fitted LabelEncoder
  try:
      encoded_wind_gust_dir = le_wind_dir.transform([wind_gust_dir_str])[0]
  except ValueError:
      print(f"Warning: Wind direction '{wind_gust_dir_str}' not seen in historical data. Defaulting to 0.")
      encoded_wind_gust_dir = 0 # Fallback value

  current_data ={
      'MinTemp': current_weather['temp_min'],
      'MaxTemp' : current_weather['temp_max'],
      'WindGustDir' : encoded_wind_gust_dir,
      'WindGustSpeed': current_weather['wind_speed'],
      'Humidity': current_weather['humidity'],
      'Pressure': current_weather['pressure'],
      'Temp': current_weather['current_temp']
  }

  # Create a DataFrame for prediction, ensuring columns match training data
  current_data_df = pd.DataFrame([current_data])

  # Predict rain tomorrow
  rain_prediction_encoded = rain_model.predict(current_data_df[X_historical.columns])
  rain_prediction_str = le_rain_tomorrow.inverse_transform(rain_prediction_encoded)[0]

  # Train regression models for Temperature and Humidity
  x_temp, y_temp = prepare_regression_data(historical_data, 'Temp')
  temp_model = train_regession_model(x_temp, y_temp)

  x_humidity, y_humidity = prepare_regression_data(historical_data, 'Humidity')
  humidity_model = train_regession_model(x_humidity, y_humidity)

  # Predict future temperature and humidity (e.g., next 5 steps/days)
  num_prediction_steps = 5

  predicted_temps = predict_future(temp_model, current_weather['current_temp'], num_steps=num_prediction_steps)
  predicted_humidity = predict_future(humidity_model, current_weather['humidity'], num_steps=num_prediction_steps)

  # Get current datetime
  now = datetime.now()
  current_datetime_str = now.strftime("%Y-%m-%d %H:%M:%S")

  # Calculate predicted dates (assuming daily steps for simplicity, consistent with historical data)
  predicted_dates = []
  for i in range(1, num_prediction_steps + 1):
      predicted_dates.append((now + timedelta(days=i)).strftime("%Y-%m-%d"))

  print(f"Current Weather in {current_weather['city']}, {current_weather['country']} on {current_datetime_str}:")
  print(f"  Temperature: {current_weather['current_temp']}°C (Feels like: {current_weather['feels_like']}°C)")
  print(f"  Min/Max Temperature: {current_weather['temp_min']}°C / {current_weather['temp_max']}°C")
  print(f"  Humidity: {current_weather['humidity']}% ({current_weather['pressure']} hPa)")
  print(f"  Wind: {current_weather['wind_speed']} m/s from {wind_gust_dir_str} ({current_weather['wind_deg']}°)")
  print(f"  Description: {current_weather['description']}")
  print(f"  Predicted Rain Tomorrow: {'Yes' if rain_prediction_str == 1 else 'No'}")

  print(f"\n--- Future Predictions for next {num_prediction_steps} days ---")
  for i in range(num_prediction_steps):
      print(f"  Date: {predicted_dates[i]}, Temperature: {predicted_temps[i]:.1f}°C, Humidity: {predicted_humidity[i]:.1f}%")
  print(f"  Note: These predictions are for the next {num_prediction_steps} daily steps based on sequential data, not a full 5-year forecast, which would require more advanced time series models.")

In [ ]:
weather_view()

Enter any city name: kyiv

--- Rain Prediction Model Evaluation ---
Accuracy: 0.8493
Precision: 0.8571
Recall: 0.3750
F1 Score: 0.5217
--------------------------------------
Current Weather in Kyiv, UA on 2026-04-01 10:45:46:
  Temperature: 13°C (Feels like: 12°C)
  Min/Max Temperature: 13°C / 13°C
  Humidity: 76% (1011 hPa)
  Wind: 4 m/s from NNW (340°)
  Description: overcast clouds
  Predicted Rain Tomorrow: No

--- Future Predictions for next 5 days ---
  Date: 2026-04-02, Temperature: 13.5°C, Humidity: 61.6%
  Date: 2026-04-03, Temperature: 11.7°C, Humidity: 56.5%
  Date: 2026-04-04, Temperature: 11.6°C, Humidity: 40.7%
  Date: 2026-04-05, Temperature: 11.7°C, Humidity: 52.9%
  Date: 2026-04-06, Temperature: 11.6°C, Humidity: 51.0%
  Note: These predictions are for the next 5 daily steps based on sequential data, not a full 5-year forecast, which would require more advanced time series models.
